# Text FTM Attack Runner (Colab + VS Code)

This notebook runs your full experiment on 100 samples with surrogate + black-box evaluation and saves results.

## 1) Prepare Project From GitHub

This step clones your repository into the Colab runtime so GPU execution can access your code and dataset.

Repository used:
- https://github.com/AbdulRehman-gitrep/ftm_text

If your repo is private, replace clone URL with your token-auth URL or use Colab GitHub auth.

In [39]:
# Run this cell first in Colab.
# It clones your GitHub repo into /content and sets FTM_PROJECT_DIR.
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/AbdulRehman-gitrep/ftm_text.git'
CLONE_PARENT = Path('/content')
CLONE_DIR = CLONE_PARENT / 'ftm_text'

if CLONE_DIR.exists():
    print('Repo already exists at', CLONE_DIR)
    # Optional refresh: fetch latest changes
    subprocess.run(['git', '-C', str(CLONE_DIR), 'pull'], check=False)
else:
    CLONE_PARENT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)

os.environ['FTM_PROJECT_DIR'] = str(CLONE_DIR)
print('FTM_PROJECT_DIR set to:', os.environ['FTM_PROJECT_DIR'])
print('Contents:', [p.name for p in CLONE_DIR.iterdir()])

Repo already exists at /content/ftm_text
FTM_PROJECT_DIR set to: /content/ftm_text
Contents: ['utils', 'requirements.txt', 'data', 'run_attack_colab.ipynb', 'config.py', 'models', '__pycache__', 'results', 'main.py', 'attacks', '.git']


In [40]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime

def is_project_dir(p: Path) -> bool:
    return (p / 'main.py').exists() and (p / 'requirements.txt').exists()

def find_project_dir():
    # Optional manual override for any environment.
    env_path = os.environ.get('FTM_PROJECT_DIR')
    if env_path:
        p = Path(env_path).expanduser()
        if is_project_dir(p):
            return p.resolve()

    # Fast checks for common local/Colab layouts.
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/FTM_text/text_ftm_attack'),
        Path('/content/drive/MyDrive/FTM_text/text_ftm_attack'),
    ]
    for p in candidates:
        if is_project_dir(p):
            return p.resolve()

    # Recursive fallback search under common roots.
    search_roots = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]
    for root in search_roots:
        if not root.exists():
            continue
        for main_file in root.rglob('main.py'):
            p = main_file.parent
            if is_project_dir(p):
                return p.resolve()

    return None

PROJECT_DIR = find_project_dir()
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find text_ftm_attack folder. Set FTM_PROJECT_DIR or place project in /content/... and rerun."
    )

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('Python =', sys.executable)

PROJECT_DIR = /content/ftm_text
Python = /usr/bin/python3


## 2) Check GPU

In [41]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    DEVICE = 'cuda:0'
else:
    DEVICE = 'cpu'
print('DEVICE =', DEVICE)

torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
DEVICE = cuda:0


## 3) Install Dependencies

In [42]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Dependencies installed.')

Dependencies installed.


## 4) Verify Dataset

In [43]:
DATA_CSV = PROJECT_DIR / 'data' / 'IMDB Dataset.csv'
if not DATA_CSV.exists():
    raise FileNotFoundError(f'Missing dataset file: {DATA_CSV}')
print('Dataset found:', DATA_CSV)

Dataset found: /content/ftm_text/data/IMDB Dataset.csv


## 5) Tuning Sweep (20 Samples Across Configs)

Run a fast sweep over multiple config_idx values, then auto-select the best config for the full run.

In [ ]:
SWEEP_TAG = datetime.now().strftime('sweep_%Y%m%d_%H%M%S')
SWEEP_ROOT = PROJECT_DIR / 'results' / SWEEP_TAG
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)

# Baseline + tuned presets from config.py
config_ids = [1, 2, 3, 4, 5, 6]
sweep_num_samples = 20

print('Sweep root:', SWEEP_ROOT)
print('Configs:', config_ids)
print('Samples per config:', sweep_num_samples)

progress_markers = [
    'Sample ',
    'Surrogate ASR',
    'Average black-box ASR',
    'Results saved to',
    'DONE',
]

sweep_rows = []
for cfg in config_ids:
    run_dir = SWEEP_ROOT / f'cfg{cfg}'
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / 'run.log'

    cmd = [
        sys.executable, 'main.py',
        '--device', DEVICE,
        '--config_idx', str(cfg),
        '--num_samples', str(sweep_num_samples),
        '--eval',
        '--data_csv', str(DATA_CSV),
        '--save_dir', str(run_dir),
    ]

    print('\n' + '=' * 70)
    print(f'SWEEP RUN: config_idx={cfg}')
    print(' '.join(cmd))

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    with open(log_path, 'w', encoding='utf-8') as log_f:
        for line in proc.stdout:
            log_f.write(line)
            if any(marker in line for marker in progress_markers):
                print(f'[cfg {cfg}] {line}', end='')

    ret = proc.wait()
    summary_path = run_dir / 'results_summary.json'

    if ret != 0 or not summary_path.exists():
        print(f'Config {cfg} failed. See {log_path}')
        continue

    with open(summary_path, 'r', encoding='utf-8') as f:
        s = json.load(f)

    row = {
        'config_idx': cfg,
        'surrogate_asr': float(s.get('surrogate_asr', 0.0)),
        'avg_black_box_asr': float(s.get('avg_black_box_asr', 0.0)),
        'avg_semantic_similarity': float(s.get('avg_semantic_similarity', 0.0)),
        'num_samples': int(s.get('num_samples', 0)),
        'summary_path': str(summary_path),
        'log_path': str(log_path),
    }
    sweep_rows.append(row)

if not sweep_rows:
    raise RuntimeError('Sweep produced no successful runs. Check sweep logs in ' + str(SWEEP_ROOT))

# Choose best config by transfer ASR first, then surrogate ASR, then similarity.
sweep_rows = sorted(
    sweep_rows,
    key=lambda r: (r['avg_black_box_asr'], r['surrogate_asr'], r['avg_semantic_similarity']),
    reverse=True,
)
BEST_CONFIG_IDX = int(sweep_rows[0]['config_idx'])

print('\n' + '=' * 70)
print('SWEEP RESULTS (sorted):')
for r in sweep_rows:
    print(
        f"cfg={r['config_idx']} | surr_asr={r['surrogate_asr']:.1f}% | "
        f"bb_asr={r['avg_black_box_asr']:.1f}% | sim={r['avg_semantic_similarity']:.4f}"
    )

print('\nSelected BEST_CONFIG_IDX =', BEST_CONFIG_IDX)
print('Top summary:', sweep_rows[0]['summary_path'])

Sweep root: /content/ftm_text/results/sweep_20260425_173725
Configs: [1, 2, 3, 4, 5]
Samples per config: 20

SWEEP RUN: config_idx=1
/usr/bin/python3 main.py --device cuda:0 --config_idx 1 --num_samples 20 --eval --data_csv /content/ftm_text/data/IMDB Dataset.csv --save_dir /content/ftm_text/results/sweep_20260425_173725/cfg1
[cfg 1] Sample 1/20
[cfg 1] Sample 2/20
[cfg 1] Sample 3/20
[cfg 1] Sample 4/20
[cfg 1] Sample 5/20
[cfg 1] Sample 6/20
[cfg 1] Sample 7/20
[cfg 1] Sample 8/20
[cfg 1] Sample 9/20
[cfg 1] Sample 10/20
[cfg 1] Sample 11/20
[cfg 1] Sample 12/20
[cfg 1] Sample 13/20
[cfg 1] Sample 14/20
[cfg 1] Sample 15/20
[cfg 1] Sample 16/20
[cfg 1] Sample 17/20
[cfg 1] Sample 18/20
[cfg 1] Sample 19/20
[cfg 1] Sample 20/20
[cfg 1]   Surrogate ASR: 15.0% (3/20)
[cfg 1]   Average black-box ASR: 10.0%
[cfg 1] Results saved to /content/ftm_text/results/sweep_20260425_173725/cfg1/
[cfg 1] DONE

SWEEP RUN: config_idx=2
/usr/bin/python3 main.py --device cuda:0 --config_idx 2 --num_sampl

In [45]:
# Full run on 100 samples using the best config from the sweep.
FULL_RUN_TAG = datetime.now().strftime('run_best_%Y%m%d_%H%M%S')
SAVE_DIR = PROJECT_DIR / 'results' / FULL_RUN_TAG
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = SAVE_DIR / 'run.log'

if 'BEST_CONFIG_IDX' not in globals():
    BEST_CONFIG_IDX = 2
    print('BEST_CONFIG_IDX not found from sweep. Falling back to config_idx=2')

cmd = [
    sys.executable, 'main.py',
    '--device', DEVICE,
    '--config_idx', str(BEST_CONFIG_IDX),
    '--num_samples', '100',
    '--eval',
    '--data_csv', str(DATA_CSV),
    '--save_dir', str(SAVE_DIR),
]

print('Running full experiment with BEST_CONFIG_IDX =', BEST_CONFIG_IDX)
print(' '.join(cmd))
print('Full logs will be saved to:', LOG_PATH)

progress_markers = [
    'Sample ',
    'iter ',
    'Surrogate ASR',
    'Average black-box ASR',
    'Results saved to',
    'DONE',
]

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True,
 )

with open(LOG_PATH, 'w', encoding='utf-8') as log_f:
    for line in proc.stdout:
        log_f.write(line)
        if any(marker in line for marker in progress_markers):
            print(line, end='')

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Full run failed with exit code {ret}. Check {LOG_PATH}')

print('Full run completed.')
print('SAVE_DIR =', SAVE_DIR)
print('LOG_PATH =', LOG_PATH)

# Artifact helper
import shutil
print('Result directory:', SAVE_DIR)
for p in sorted(Path(SAVE_DIR).glob('*')):
    print('-', p.name)

zip_path = str(SAVE_DIR) + '.zip'
if not Path(zip_path).exists():
    shutil.make_archive(str(SAVE_DIR), 'zip', root_dir=str(SAVE_DIR))
print('ZIP:', zip_path)

drive_root = Path('/content/drive/MyDrive')
if drive_root.exists():
    dest_dir = drive_root / 'ftm_results'
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / Path(zip_path).name
    shutil.copy2(zip_path, dest)
    print('Copied zip to Drive:', dest)
else:
    print('Drive not mounted. To persist, mount Drive then copy the zip manually.')

Running full experiment with BEST_CONFIG_IDX = 1
/usr/bin/python3 main.py --device cuda:0 --config_idx 1 --num_samples 100 --eval --data_csv /content/ftm_text/data/IMDB Dataset.csv --save_dir /content/ftm_text/results/run_best_20260425_174133
Full logs will be saved to: /content/ftm_text/results/run_best_20260425_174133/run.log
Sample 1/100
  iter  100/300 | pred=1 target=0 | changed=0.3% | logit[target]=3.951
  iter  200/300 | pred=1 target=0 | changed=0.3% | logit[target]=3.987
Sample 2/100
  iter  100/300 | pred=1 target=0 | changed=0.2% | logit[target]=4.205
  iter  200/300 | pred=1 target=0 | changed=0.5% | logit[target]=4.170
Sample 3/100
  iter  100/300 | pred=0 target=1 | changed=1.2% | logit[target]=4.253
  iter  200/300 | pred=0 target=1 | changed=1.6% | logit[target]=4.244
Sample 4/100
  iter  100/300 | pred=0 target=1 | changed=0.4% | logit[target]=3.829
  iter  200/300 | pred=0 target=1 | changed=0.8% | logit[target]=3.820
Sample 5/100
  iter  100/300 | pred=0 target=1 | c

## 6) Load and Print Final Metrics

In [46]:
summary_path = SAVE_DIR / 'results_summary.json'
examples_path = SAVE_DIR / 'examples.txt'

if not summary_path.exists():
    raise FileNotFoundError(f'Summary not found: {summary_path}')

with open(summary_path, 'r', encoding='utf-8') as f:
    summary = json.load(f)

print('===== FINAL METRICS =====')
print('num_samples:', summary.get('num_samples'))
print('surrogate_asr:', summary.get('surrogate_asr'))
print('avg_semantic_similarity:', summary.get('avg_semantic_similarity'))
print('above_sim_threshold:', summary.get('above_sim_threshold'))

bb = summary.get('black_box_asr', {})
if bb:
    print('black_box_asr:')
    for k, v in bb.items():
        print(' -', k, ':', v)
    print('avg_black_box_asr:', summary.get('avg_black_box_asr'))

print('summary_path =', summary_path)
print('examples_path =', examples_path)

===== FINAL METRICS =====
num_samples: 100
surrogate_asr: 6.0
avg_semantic_similarity: 0.9986325013637543
above_sim_threshold: 100
black_box_asr:
 - textattack/bert-base-uncased-imdb : 3.0
 - textattack/roberta-base-imdb : 3.0
 - textattack/xlnet-base-cased-imdb : 3.0
avg_black_box_asr: 3.0
summary_path = /content/ftm_text/results/run_best_20260425_174133/results_summary.json
examples_path = /content/ftm_text/results/run_best_20260425_174133/examples.txt


## 7) Optional: Zip Results for Download

In [47]:
import shutil
zip_path = str(SAVE_DIR) + '.zip'
shutil.make_archive(str(SAVE_DIR), 'zip', root_dir=str(SAVE_DIR))
print('Created:', zip_path)

Created: /content/ftm_text/results/run_best_20260425_174133.zip
